In [ ]:
import scipp as sc
import scippnexus as snx
import plopp as pp

from easydynamics.Job import Job
from easydynamics.experiment import Experiment
from easydynamics.experiment import Data

# from easydynamics.sample import BrownianTranslationalDiffusion
from easydynamics.sample import JumpDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import Lorentzian
from easydynamics.sample import DeltaFunction
from easydynamics.sample import Polynomial
# from easydynamics.sample import DampedHarmonicOscillator


from easydynamics.sample import Gaussian

%matplotlib widget
data_path = r"C:\Users\henrikjacobsen3\Dropbox\DMSC\Halric 2025\new_IRIS_data\iris_Cells_Plus_TCZ_108330_to108389_Plus_Empty_Data_SQW"



In [ ]:
# Load data function
def load_data(filenumber: int):
    filename = data_path + fr"/iris{filenumber}_graphite002_sqw.nxs"
    all_data = snx.load(filename)

    temperature = all_data['mantid_workspace_1']['logs']['Sample']
    temperature.coords['time'].unit = 's'
    temperature.unit = 'K'

    data=all_data['mantid_workspace_1']['workspace'].rename({'axis1': 'energy', 'axis2': 'Q'})

    data.coords['energy'].unit = 'meV'
    data.coords['Q'].unit = '1/Angstrom'

    del data.coords['frac_area']
    return data, temperature




In [ ]:
filenumber = 108556 # Empty container long
empty_data1, empty_temperature1 = load_data(filenumber)
filenumber = 108557 # Empty container long2
empty_data2, empty_temperature2 = load_data(filenumber)

empty_data = (empty_data1 + empty_data2) / 2
# pp.slicer(empty_data,vmin=0, vmax=1)

In [ ]:
# Let's start with just one data set
filenumber = 108331 # base T long
lowt_data,lowt_temperature = load_data(filenumber)

lowt_data_shorter, lowt_temperature_shorter = load_data(108330)

filenumber = 108388 #290 K long
highT_data, data_temperature = load_data(filenumber)


In [ ]:
pp.slicer(lowt_data,vmin=0, vmax=8)

In [ ]:
# The data clearly needs to be rebinned
pp.slicer(highT_data)

In [ ]:
# Rebin data and use midpoints instead of edges
number_of_Q_bins = 10 # we may change this later, but this seems like a reasonable start
number_of_energy_bins = 501
rebinned_data=highT_data.rebin(Q=number_of_Q_bins, energy=number_of_energy_bins)
rebinned_lowt_data=lowt_data.rebin(Q=number_of_Q_bins, energy=number_of_energy_bins)

rebinned_lowt_data.coords['Q'] = sc.midpoints(rebinned_lowt_data.coords['Q'])
rebinned_lowt_data.coords['energy'] = sc.midpoints(rebinned_lowt_data.coords['energy'])
rebinned_data.coords['Q'] = sc.midpoints(rebinned_data.coords['Q'])
rebinned_data.coords['energy'] = sc.midpoints(rebinned_data.coords['energy'])


# Remove 0 variances
v = rebinned_data.variances
v[v <= 0] = 1.0
v = rebinned_lowt_data.variances
v[v <= 0] = 1.0
pp.slicer(rebinned_data)

In [ ]:
pp.slicer(rebinned_lowt_data)

In [ ]:
# Now let's use the low temperature data to estimate the resolution.
resolution_job= Job(name='resolution')

exp=Experiment()
res_data=Data()
res_data.append(rebinned_lowt_data)

exp.set_data(res_data)

resolution_job.set_experiment(exp)


bg=SampleModel('Background')
bg.add_component(Polynomial(coefficients=[1e-3]))
resolution_job.set_background_model(bg)


resolution_model=SampleModel(name="ResolutionModel")
resolution_model.add_component(Gaussian(name="Res1", area=1.5,width=0.01))
resolution_model.add_component(Gaussian(name="Res2", area=1.0,width=0.015,center=-0.01))
resolution_model.add_component(Lorentzian(name="Res3", area=0.3,width=0.015,center=-0.025))

resolution_job.set_theory(resolution_model)
resolution_job.generate_analysis_for_cuts()
for i in range(len(resolution_job.analysis)):
    resolution_job.analysis[i].get_fit_parameters()[8].min=0 #polynomial
    resolution_job.analysis[i].get_fit_parameters()[0].min=0 #Gauss 1 area
    resolution_job.analysis[i].get_fit_parameters()[2].min=0 #Gauss 2 area

resolution_job.plot_data_and_model(intensity_min=0.0, intensity_max=20.0,
                            energy_min=-0.5, energy_max=1.3)

In [ ]:
resolution_job.fit(sequential = "Q")

resolution_job.plot_data_and_model(intensity_min=0.0, intensity_max=120.0,
                            energy_min=-0.2, energy_max=0.2)

In [ ]:
Cells_Plus_TCZ= Job(name='Cells_Plus_TCZ')

exp=Experiment()
highT_data_ED=Data()
highT_data_ED.append(rebinned_data)
exp.set_data(highT_data_ED)
Cells_Plus_TCZ.set_experiment(exp)


bg=SampleModel('Background')
bg.add_component(Polynomial(name='BG',coefficients=[1e-3]))
Cells_Plus_TCZ.set_background_model(bg)

Cells_Plus_TCZ_model=SampleModel(name="Cells_Plus_TCZ_model")
Cells_Plus_TCZ_model.add_component(Lorentzian(name="QuasiElastic", area=0.3, width=0.1))
Cells_Plus_TCZ_model.add_component(DeltaFunction(name="Elastic", area=0.1))
Cells_Plus_TCZ.set_theory(Cells_Plus_TCZ_model)

Cells_Plus_TCZ.generate_analysis_for_cuts()

Cells_Plus_TCZ.use_fit_as_resolution(resolution_job)

for i in range(len(Cells_Plus_TCZ.analysis)):
    Cells_Plus_TCZ.analysis[i].get_fit_parameters()[3].min=0 #polynomial



In [ ]:

Cells_Plus_TCZ.fit()
Cells_Plus_TCZ.plot_data_and_model_residual(intensity_min=-0.5, intensity_max=5,
                            energy_min=-0.3, energy_max=0.5)

In [ ]:
Cells_Plus_TCZ.plot_fit_parameters("QuasiElastic width")

In [ ]:
Cells_Plus_TCZ.plot_fit_parameters("QuasiElastic area")

In [ ]:
Cells_Plus_TCZ.plot_fit_parameters("Elastic area")

In [ ]:
parameters = Cells_Plus_TCZ.get_parameters_as_data_group()

In [ ]:
normalised_elastic = parameters['Elastic area']['value'] / (parameters['Elastic area']['value'] + parameters['QuasiElastic area']['value'])
sc.plot(normalised_elastic)

In [ ]:
sum_of_peaks = (parameters['Elastic area']['value'] + parameters['QuasiElastic area']['value'])
sc.plot(sum_of_peaks)

In [ ]:
Cells_Plus_TCZ._diffusion_model=JumpDiffusion(name="DiffusionModel", diffusion_coefficient=0.0823,tau = 1, scale=1.0)
fit_result=Cells_Plus_TCZ.fit_jump_diffusion_width("QuasiElastic width")
Cells_Plus_TCZ.plot_diffusion_fit_result("QuasiElastic width")


In [ ]:
Cells_Plus_TCZ._diffusion_model.get_parameters()

In [ ]:


Cells_Plus_TCZ_simultaneous= Job(name='JumpDiffusion')


exp=Experiment()
highT_data_simul=Data()
highT_data_simul.append(rebinned_data)

exp.set_data(highT_data_simul)

Cells_Plus_TCZ_simultaneous.set_experiment(exp)
Cells_Plus_TCZ_simultaneous.generate_empty_analysis_array()


bg=SampleModel('Background')
bg.add_component(Polynomial(coefficients=[1e-3]))
Cells_Plus_TCZ_simultaneous.set_background_model(bg)
Cells_Plus_TCZ_simultaneous.set_background_model_for_all_analyses()


diffusion_model=JumpDiffusion(name="JumpDiffusion", diffusion_coefficient=1.8, tau = 1.0,scale=0.4)
Cells_Plus_TCZ_simultaneous.set_diffusion_model(diffusion_model)
# delta_model=DeltaFunction(name="Delta",area=0.05)
# Cells_Plus_TCZ_simultaneous.set_theory_for_all_analyses(delta_model) # Something wrong here, it uses the same parameter
# Cells_Plus_TCZ.use_fit_as_resolution(resolution_job) # doesn't seem to work for some reason
for i in range(len(Cells_Plus_TCZ_simultaneous._analysis)):
    this_resolution_model=resolution_job._analysis[i]._theory
    Cells_Plus_TCZ_simultaneous._analysis[i].set_resolution_model(this_resolution_model)
    Cells_Plus_TCZ_simultaneous._analysis[i].fix_resolution_parameters()
    Cells_Plus_TCZ_simultaneous._analysis[i]._theory.add_component(DeltaFunction(name="Delta",area=0.05))
    Cells_Plus_TCZ_simultaneous.analysis[i].get_fit_parameters()[5].min=0.0


Cells_Plus_TCZ_simultaneous.plot_data_and_model_residual(intensity_min=-0.5, intensity_max=5,
                            energy_min=-0.3, energy_max=0.5)

In [ ]:
result=Cells_Plus_TCZ_simultaneous.fit_simultaneous()

Cells_Plus_TCZ_simultaneous.plot_data_and_model_residual(intensity_min=-0.5, intensity_max=5,
                            energy_min=-0.3, energy_max=0.5)

In [27]:
Cells_Plus_TCZ_simultaneous.analysis[1].get_fit_parameters()

[<Parameter 'scale': 0.4609, bounds=[-inf:inf]>,
 <Parameter 'Delta area': 0.1011 meV, bounds=[0.0:inf]>,
 <Parameter 'diffusion_coefficient': 1.8800, bounds=[-inf:inf]>,
 <Parameter 'tau': 1.5219, bounds=[-inf:inf]>,
 <Parameter 'scale': 0.4609, bounds=[-inf:inf]>,
 <Parameter 'Polynomial_c0': 0.0076, bounds=[0.0:inf]>,
 <Parameter 'offset': -0.0034 meV, bounds=[-inf:inf]>]

In [ ]:
Cells_Plus_TCZ_simultaneous.plot_fit_parameters("Delta area")
# Cells_Plus_TCZ_simultaneous